# M5. RAG 체인 (05_m5_RAG체인)

질문 → 관련 사고사례 검색 → 프롬프트 구성 → LLM 답변 → 출처 표시까지 연결합니다.

| 체크리스트 | 이 노트북의 위치 |
|---|---|
| M5-1 사용자 질문 입력 | `ask(question)` 함수의 입력 (4절) |
| M5-2 관련 문서 검색 | `retrieve()` (2절) |
| M5-3 검색 결과를 프롬프트에 전달 | `build_prompt()` (3절) |
| M5-4 LLM 답변 생성 | `ask()` 안의 LLM 호출 (4절) |
| M5-5 출처 표시와 근거 부족 처리 | `MIN_SIM` 기준 + 출처 목록 (2·4절) |
| M5-6 질문부터 답변까지 전체 테스트 | 5절 |

**사전 준비**: 04 노트북으로 `data/chroma`에 6,032건이 적재되어 있어야 하고, `.env`에 `OPENAI_API_KEY`가 있어야 합니다.

## 필요 패키지
이 노트북은 **chromadb, openai, python-dotenv, pandas** 를 사용합니다. 프로젝트 환경(uv)에 설치되어 있어야 합니다.
`ModuleNotFoundError`가 나면 터미널에서 아래처럼 설치한 뒤 커널을 재시작하세요.
```
uv add chromadb openai python-dotenv pandas
```

## 0. 준비

In [3]:
import os
from pathlib import Path
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

# data/chroma가 있는 폴더를 위로 올라가며 찾는다
ROOT = Path.cwd()
while not (ROOT / "data" / "chroma").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "chroma").exists(), "data/chroma가 없습니다. 04 노트북을 먼저 실행하세요."

load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), f"{ROOT / '.env'} 에 OPENAI_API_KEY=키값 을 넣고 커널을 재시작하세요."
client = OpenAI(timeout=60, max_retries=2)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")
K = 3             # 검색해 올 사례 수
MIN_SIM = 0.40    # 이 값보다 비슷하지 않은 사례는 근거로 쓰지 않는다 (5-1절 점검: 범위 밖 최고 0.353 < 0.40 < 범위 안 최저 0.438)
NO_EVIDENCE = "제공된 사례에서 근거를 찾을 수 없습니다."

col = chromadb.PersistentClient(path=str(ROOT / "data" / "chroma")).get_collection("sif")
assert col.count() > 0, "sif 컬렉션이 비어 있습니다. 04 노트북을 먼저 실행하세요."
print("프로젝트 폴더:", ROOT)
print("컬렉션 문서 수:", col.count(), "| 답변 모델:", CHAT_MODEL)

프로젝트 폴더: /Users/song/Desktop/unit1_project
컬렉션 문서 수: 6032 | 답변 모델: gpt-4o-mini


## 1. 설계
1. 질문을 임베딩해서 가장 비슷한 사례 K건을 찾는다 (필터로 업종 등을 지정할 수 있다).
2. 유사도가 `MIN_SIM`보다 낮은 사례는 버린다. **남은 사례가 없으면 LLM을 부르지 않고** "근거를 찾을 수 없습니다"로 답한다.
3. 남은 사례를 `[사례1]`, `[사례2]`… 형식으로 프롬프트에 넣는다.
4. LLM은 사례 안의 내용만으로 `1) 사고 원인 2) 예방 대책`을 답하고, 근거가 되는 `[사례N]`을 표시한다.
5. 답변 아래에 출처(업종·기인물·재해연도·개요)를 붙인다.

## 2. 관련 문서 검색 (M5-2, M5-5)

In [4]:
def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=[text[:6000]]).data[0].embedding

def retrieve(question, k=K, where=None, min_sim=None):
    """질문과 비슷한 사례를 유사도 순으로 돌려준다. min_sim 미만은 제외."""
    min_sim = MIN_SIM if min_sim is None else min_sim
    res = col.query(query_embeddings=[embed(question)], n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        sim = 1 - dist
        if sim >= min_sim:
            hits.append({"similarity": round(sim, 3), "document": doc, "meta": meta})
    return hits

# 동작 확인
for h in retrieve("지게차 작업 중 협착 사고"):
    print(h["similarity"], h["meta"]["업종구분"], h["meta"]["기인물"], "|", h["document"].split("[재해개요]")[-1][:50].replace("\n", " "))

0.557 제조업등 지게차 |  2016년 9월경 OO사업장 제품 하역장에서 재해자가 작업 중 하역작업을 하기 위해 후진
0.551 제조업등 지게차 |  2018년 12월경 야적장에서 바닥에 철근 원곡 라인에 가공한 철근을 확인하기 위해 앉아
0.549 제조업등 지게차 |  2021년 12월경 자동차 부품 창고에서 재해자 2명이 재고조사를 위해 지게차 포크에 작


## 3. 프롬프트 구성 (M5-3)

In [5]:
SYSTEM = """당신은 산업재해 예방을 돕는 안전 어시스턴트입니다. 아래 [사례]에 적힌 내용만 근거로 답하세요.
- 답변은 '1) 사고 원인', '2) 예방 대책' 순서로 쓰고, 각 항목마다 근거가 된 사례 번호를 [사례N] 형식으로 표시하세요.
- 사례에 없는 내용은 추측하거나 지어내지 마세요. 법령 조항 번호도 사례에 없으면 쓰지 마세요.
- 사례만으로 질문에 답할 수 없으면 다른 말 없이 정확히 '제공된 사례에서 근거를 찾을 수 없습니다.'라고만 답하세요."""

def build_prompt(question, hits):
    blocks = []
    for n, h in enumerate(hits, 1):
        m = h["meta"]
        head = f"[사례{n}] (업종구분: {m['업종구분']}, 기인물: {m['기인물'] or '미분류'}, 재해연도: {m['재해연도']})"
        blocks.append(head + "\n" + h["document"])
    context = "\n\n".join(blocks)
    return f"{context}\n\n질문: {question}"

demo = retrieve("비계에서 추락한 사고 대책", where={"업종구분": "건설업"})
print(build_prompt("비계에서 추락한 사고 대책", demo)[:800])

[사례1] (업종구분: 건설업, 기인물: 비계, 재해연도: 2021)
[업종] 건설업 > 4. 마감공사 > 4.4 도장작업 > 4.4.3 실외 도장
[재해종류] 추락
[재해개요] 2021년 12월경 ○○현장에서 피재자가 비계 3단 작업발판에서 비계 2단 상부안전난간대를 밟고 내려오던 중 미끄러지면서 비계 3단 작업발판과 2단 상부안전난간대 사이로 추락하여 사망
[기인물] 비계
[재해유발요인] 도장작업을 위해 외부비계 작업발판에서 비계 외측 상부안전난간대를 밟고 내려오던 중 미끄러지면서 추락
[예방대책] ▶ 비계 내측에 근로자가 상, 하로 이동할 수 있는 계단식 구조의 발판을 설치하고 개방된 양 측면에는 안전난간을 2단으로 구성하여 전용 이동통로를 이용하도록 관리감독
▶ 계단형 발판을 설치할 수 없을 경우, 해치형 작업발판 및 사다리 이동통로를 구성하여 근로자가 비계 외측으로 이동하는 것을 금지 조치

[사례2] (업종구분: 건설업, 기인물: 비계, 재해연도: 2019)
[업종] 건설업 > 13. 기타 건설공사 > 13.1 안전가시설 작업 > 13.1.3 안전가시설(비계)
[재해종류] 추락
[재해개요] 2019년 1월경 ○○현장에서 재해자가 건물 내부 시스템비계 6단에서 상부에 해체된 거푸집(유로폼) 정리작업 완료 후 시스템비계 외부를 통하여 지상으로 내려오던 중, 몸의 중심을 잃고 지상1층 바닥(H≒8m)으로 떨여져 사망
[기인물] 비계
[재해유발요인] 승강통로가 미설치된 상태에서 외부비계 바깥으로 이동 중 실족하여 추락
[예방대책] ▶ 비계 내측에 근로자가 상, 하로 이동할 수 있는 계단식 구조의 발판을 설


## 4. 답변 생성과 출처 표시 (M5-1, M5-4, M5-5)
`ask(질문)`이 전체를 연결합니다. 결과는 `answer`(답변), `sources`(출처), `grounded`(사례에 근거해 답했는지)를 담은 딕셔너리입니다.

In [6]:
def ask(question, k=K, where=None):
    hits = retrieve(question, k=k, where=where)
    sources = [{
        "사례": n,
        "유사도": h["similarity"],
        "업종구분": h["meta"]["업종구분"],
        "중분류": h["meta"]["중분류"],
        "기인물": h["meta"]["기인물"] or "미분류",
        "재해연도": h["meta"]["재해연도"],
        "개요": h["document"].split("[재해개요]")[-1].split("\n[기인물]")[0].strip()[:80],
    } for n, h in enumerate(hits, 1)]

    if not hits:                       # 근거가 될 사례가 없으면 LLM을 부르지 않는다
        return {"answer": NO_EVIDENCE, "sources": [], "grounded": False, "retrieved": []}

    resp = client.chat.completions.create(
        model=CHAT_MODEL, temperature=0,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": build_prompt(question, hits)}],
    )
    answer = resp.choices[0].message.content.strip()
    grounded = NO_EVIDENCE not in answer
    return {"answer": answer, "sources": sources if grounded else [], "grounded": grounded, "retrieved": sources}

def show(question, **kw):
    r = ask(question, **kw)
    print("Q.", question)
    print(r["answer"])
    if r["sources"]:
        print("\n출처")
        for s in r["sources"]:
            print(f"  [사례{s['사례']}] 유사도 {s['유사도']} | {s['업종구분']} > {s['중분류']} | 기인물 {s['기인물']} | {s['재해연도']}년 | {s['개요']}")
    if not r["grounded"] and r["retrieved"]:
        names = ", ".join(x["기인물"] for x in r["retrieved"])
        print(f"(검색은 됐지만 질문과 맞지 않아 답하지 않음 | 검색된 기인물: {names})")
    print("근거 있음:", r["grounded"])
    print("=" * 80)
    return r

_ = show("지게차 작업 중 협착 사고는 왜 발생하고 어떻게 예방하나요?")

Q. 지게차 작업 중 협착 사고는 왜 발생하고 어떻게 예방하나요?
1) 사고 원인: 지게차 작업 중 협착 사고는 작업자 간 의사소통이 이루어지지 않고 작업을 실시하여 발생하며, 이로 인해 작업 중인 기계 또는 운반하는 물체에 부딪히거나 깔림, 주변 구조물과 운반하는 물체 사이에 끼임 등의 위험이 존재합니다. [사례3]

2) 예방 대책: 지게차 작업 시에는 시야 확보 후 작업을 실시해야 하며, 시야 확보가 어려운 경우에는 2인 1조로 인원 편성 후 작업지휘자를 지정하여 관리해야 합니다. [사례3]

출처
  [사례1] 유사도 0.513 | 제조업등 > 철도·항공·창고·운수관련서비스업 | 기인물 지게차 | 2021년 | 2021년 12월경 자동차 부품 창고에서 재해자 2명이 재고조사를 위해 지게차 포크에 작업케이지를 연결하여 상승시킨 후 작업하던 중 약 7m 높
  [사례2] 유사도 0.512 | 제조업등 > 기계기구·금속·비금속광물제품제조업 | 기인물 지게차 | 2018년 | 2018년 12월경 야적장에서 바닥에 철근 원곡 라인에 가공한 철근을 확인하기 위해 앉아 있던 중, 이를 확인하지 못한 동료가 지게차를 이용하여
  [사례3] 유사도 0.499 | 제조업등 > 철도·항공·창고·운수관련서비스업 | 기인물 지게차 | 2017년 | 2017년 7월결 야적장에서 지게차 운전자가 창고의 제품을 컨테이너로 옮기기 위해 지게차 작업을 하던 중, 지게차와 재해자가 충돌하여 재해자의 
근거 있음: True


## 5. 전체 테스트 (M5-6)
- **범위 안 질문**: 사례집에 있을 법한 질문 → 근거와 출처가 붙어야 합니다.
- **범위 밖 질문**: 사고 사례와 무관한 질문 → "근거를 찾을 수 없습니다"가 나와야 합니다.
- `where`로 업종 필터를 건 질문도 확인합니다.

In [7]:
in_scope = [
    "컨베이어에 끼인 사고는 어떻게 예방하나요?",
    "천장크레인으로 중량물을 옮기다 발생한 사고의 원인과 대책은?",
    "이동식 사다리에서 떨어진 사고를 막으려면?",
]
for q in in_scope:
    show(q)

# 업종 필터(건설업)를 건 질문
show("비계에서 추락한 사고를 예방하려면?", where={"업종구분": "건설업"})

Q. 컨베이어에 끼인 사고는 어떻게 예방하나요?
1) 사고 원인: 컨베이어에 끼인 사고는 설비에 안전장치가 설치되어 있지 않거나 불량하거나, 설치된 안전장치를 해제한 상태로 사용 중 발생하는 위험이 원인입니다 [사례1]. 또한, 설비를 정지시키지 않거나 전원을 차단하지 않은 상태에서 정비·보수·청소·교체·조정·점검 등의 작업 중에도 사고가 발생할 수 있습니다 [사례3].

2) 예방 대책: 컨베이어 벨트와 테일 풀리 사이 끼임의 위험이 있는 부분에 견고한 구조의 방호덮개 또는 울을 설치하고, 벨트 컨베이어 옆에는 근로자가 안전하게 통행할 수 있는 안전통로를 확보해야 합니다 [사례1]. 또한, 컨베이어의 운전을 즉시 정지시킬 수 있도록 비상정지장치를 설치하고, 근로자에게 작업에 알맞은 작업복을 착용토록 하여 복장을 단정히 관리해야 합니다 [사례3].

출처
  [사례1] 유사도 0.47 | 제조업등 > 시설관리및사업지원서비스업 | 기인물 컨베이어 | 2019년 | 2019년 11월경 재해자가 컨베이어 헤드 풀리의 이물질을 제거 후 설비의 전원을 켜고 위치로 이동하던 중, 컨베이어 테일 풀리 부위에 끼여 사
  [사례2] 유사도 0.467 | 제조업등 > 목재및종이제품제조업 | 기인물 컨베이어 | 2016년 | 2016년 11월 목재가공공장에서 페이로더 운전원인 재해자가 호퍼 주변 작업 중인 타 작업자의 업무상황을 확인하기 위하여 컨베이어 상부에 올라 
  [사례3] 유사도 0.464 | 제조업등 > 기계기구·금속·비금속광물제품제조업 | 기인물 컨베이어 | 2016년 | 2016년 12월경 규사원석 분쇄작업장에서 컨베이어를 점검하던 중 컨베이어벨트와 리턴드럼 사이에 신체일부가 끼어 사망
근거 있음: True
Q. 천장크레인으로 중량물을 옮기다 발생한 사고의 원인과 대책은?
1) 사고 원인: 운반하는 화물의 형태에 적합하지 않은 줄걸이 방법 또는 불량한 작업방법으로 화물의 균형이 맞지 않은 상태에서 무리하게 인양하다가 화물이 떨어지면서 맞거나 그 아래 깔림 위험이 

{'answer': '1) 사고 원인  \n- 승강통로가 미설치된 상태에서 외부비계 바깥으로 이동 중 실족하여 추락 [사례1]  \n- 비계 수평재를 밟고 비계 해체 작업 중 몸의 중심을 잃고 바닥으로 추락 [사례2]  \n- 작업발판 3단에서 비계 외측 수평재를 밟고 내려오던 중 지면 바닥으로 떨어짐 [사례3]  \n\n2) 예방 대책  \n- 비계 내측에 근로자가 상, 하로 이동할 수 있는 계단식 구조의 발판을 설치하고 개방된 양 측면에는 안전난간을 2단으로 구성하여 전용 이동통로를 이용하도록 관리감독 [사례1, 사례3]  \n- 계단형 발판을 설치할 수 없을 경우, 해치형 작업발판 및 사다리 이동통로를 구성하여 근로자가 비계 외측으로 이동하는 것을 금지 조치 [사례1, 사례3]  \n- 비계 조립·해체작업시 선행 안전난간대 설치 적용 검토 [사례2]  \n- 비계 조립·해체 작업하는 경우 폭20cm 이상의 발판을 설치 및 안전난간 설치 [사례2]  \n- 안전난간 설치가 곤란한 경우 안전대의 부착설비 설치 [사례2]  \n- 근로자에게 안전대를 착용하도록 하는 등 떨어짐 방지조치 실시 [사례2]',
 'sources': [{'사례': 1,
   '유사도': 0.478,
   '업종구분': '건설업',
   '중분류': '건설업',
   '기인물': '비계',
   '재해연도': 2019,
   '개요': '2019년 1월경 ○○현장에서 재해자가 건물 내부 시스템비계 6단에서 상부에 해체된 거푸집(유로폼) 정리작업 완료 후 시스템비계 외부를 통하여 '},
  {'사례': 2,
   '유사도': 0.464,
   '업종구분': '건설업',
   '중분류': '건설업',
   '기인물': '비계',
   '재해연도': 2016,
   '개요': '2016년 10월경 ○○현장에서 외부 비계 수평재를 밟고 비계 해체 작업 중 몸의 중심을 잃고 바닥으로 추락하여 사망'},
  {'사례': 3,
   '유사도': 0.464,
   '업종구분': '건설업',
   '중

In [8]:
out_of_scope = [
    "오늘 서울 날씨 알려줘",
    "달 탐사선은 언제 발사되나요?",
    "맛있는 파스타 레시피 알려줘",
]
for q in out_of_scope:
    show(q)

Q. 오늘 서울 날씨 알려줘
제공된 사례에서 근거를 찾을 수 없습니다.
근거 있음: False
Q. 달 탐사선은 언제 발사되나요?
제공된 사례에서 근거를 찾을 수 없습니다.
근거 있음: False
Q. 맛있는 파스타 레시피 알려줘
제공된 사례에서 근거를 찾을 수 없습니다.
근거 있음: False


### 5-0. 알려진 한계: 구체 용어가 일반 표현에 묻히는 경우
질문에 구체 용어(예: `지붕 채광판`)와 일반 표현(예: `작업하다 추락한 사고`)이 섞여 있으면, 임베딩 검색이 일반 표현 쪽 사례(거푸집·작업발판 등)를 가져올 수 있습니다.
그러면 LLM은 사례가 질문과 맞지 않다고 보고 "근거를 찾을 수 없습니다"로 답합니다. 틀린 답을 지어내지 않는 대신 답을 못 하는 경우입니다.
→ 개선 후보(M7): 질문에서 기인물을 뽑아 메타데이터 필터로 쓰기 / 키워드 검색과 결합하기. 아래에서 같은 주제를 두 가지 표현으로 비교합니다.

In [9]:
for q in ["지붕 채광판 위에서 작업하다 추락한 사고 대책", "지붕 채광판 파손 추락"]:
    show(q, where={"업종구분": "건설업"})

Q. 지붕 채광판 위에서 작업하다 추락한 사고 대책
제공된 사례에서 근거를 찾을 수 없습니다.
(검색은 됐지만 질문과 맞지 않아 답하지 않음 | 검색된 기인물: 거푸집, 작업발판/가설계단, 작업발판/가설계단)
근거 있음: False
Q. 지붕 채광판 파손 추락
1) 사고 원인: 지붕에서 이동하던 중 밟고 있던 마감재(선라이트 또는 슬레이트)가 파손되면서 바닥으로 추락 [사례1, 사례2, 사례3].

2) 예방 대책: ▶ 노후화 되거나 강도가 약한 재료의 지붕재 위에 지붕 전용 안전덮개(발판)를 견고하게 설치 ▶ 추락방지망 설치, 안전대 부착설비 설치 등의 안전 방호 조치 실시 ▶ 충분한 강도와 규격으로 제작된 지붕사다리 설치 및 근로자 이동 또는 작업 시 안전대 체결 ▶ 지붕재의 해체 또는 설치 작업 시, 고소작업대 등을 사용하여 지붕 위에서의 작업을 최소화 [사례1, 사례2, 사례3].

출처
  [사례1] 유사도 0.556 | 건설업 > 건설업 | 기인물 지붕 채광판 등(선라이트, 슬레이트) | 2019년 | 2019년 10월경 ○○현장에서 피재자가 지붕위로 이동 중 투명 PC판넬을 밟아 파손되면서 지붕에서 약8m 하부 콘크리트 바닥으로 떨어져 사망
  [사례2] 유사도 0.554 | 건설업 > 건설업 | 기인물 지붕 채광판 등(선라이트, 슬레이트) | 2021년 | 2021년 06월경 지붕의 채광창 보수작업을 위한 사전 공사개소 확인을 위해 지붕으로 이동하면서 밟은 노후된 샌드위치패널이 파손되면서 약 7.4
  [사례3] 유사도 0.543 | 건설업 > 건설업 | 기인물 지붕 채광판 등(선라이트, 슬레이트) | 2019년 | 2019년 6월경 재해자가 물품보관 창고에서 선라이트 지붕 위 C형강에 컬러강판을 덧씌우기 위해 반대쪽 지붕에 적재된 컬러강판을 가지러 이동하던
근거 있음: True


### 5-1. 유사도 점검 (MIN_SIM 조정용)

In [10]:
import pandas as pd

def top_sim(q):
    res = col.query(query_embeddings=[embed(q)], n_results=1)
    return round(1 - res["distances"][0][0], 3)

rows = [{"구분": "범위 안", "질문": q, "최고 유사도": top_sim(q)} for q in in_scope] + \
       [{"구분": "범위 밖", "질문": q, "최고 유사도": top_sim(q)} for q in out_of_scope]
chk = pd.DataFrame(rows)
display(chk)
lo_in = chk[chk["구분"] == "범위 안"]["최고 유사도"].min()
hi_out = chk[chk["구분"] == "범위 밖"]["최고 유사도"].max()
print(f"범위 안 최저 {lo_in} / 범위 밖 최고 {hi_out} / 현재 MIN_SIM = {MIN_SIM}")
print("→ MIN_SIM은 범위 밖 최고값보다 크고 범위 안 최저값보다 작아야 합니다.")

,구분,질문,최고 유사도
0,범위 안,컨베이어에 끼인 사고는 어떻게 예방하나요?,0.470
1,범위 안,천장크레인으로 중량물을 옮기다 발생한 사고의 원인과 대책은?,0.459
2,범위 안,이동식 사다리에서 떨어진 사고를 막으려면?,0.438
3,범위 밖,오늘 서울 날씨 알려줘,0.278
4,범위 밖,달 탐사선은 언제 발사되나요?,0.353
5,범위 밖,맛있는 파스타 레시피 알려줘,0.230


범위 안 최저 0.438 / 범위 밖 최고 0.353 / 현재 MIN_SIM = 0.4
→ MIN_SIM은 범위 밖 최고값보다 크고 범위 안 최저값보다 작아야 합니다.


## 정리
- 질문 → 검색(K=3, `MIN_SIM` 기준) → 사례 기반 프롬프트 → 답변 + 출처까지 `ask()` 한 번으로 동작합니다.
- 근거가 없으면 LLM을 호출하지 않고 "제공된 사례에서 근거를 찾을 수 없습니다."로 답합니다.
- 다음 단계(M6): 평가 질문 세트를 만들어 검색 품질(Hit@K, MRR)을 측정합니다.